In [ ]:
import torch
import numpy as np
from datetime import datetime
from .agent import Agent
from .active_learning import ActiveLearningPipeline
from .llm_advisor import LLMAdvisor, RewardShaper, ALAdvisor
from .vae_detector import VAEAnomalyDetector, VAEAdapter
from .lstm_qnetwork import LSTMAgent
import os

The Environment class contains the complete workflow which includes
Data loading and preprocessing, model definition and configuration, training pipeline and optimization, evaluation metrics and performance analysis and finally visualization of the results with proper insights.

In [ ]:
class Environment:
    def __init__(self, config):
        self.config = config

        # Initialize agent
        self.agent = Agent(
            config['model']['state_size'],
            config['model']['action_size']
        )

        # Initialize VAE
        self.vae = VAEAnomalyDetector(
            input_dim=config['model']['state_size'],
            latent_dim=4
        )
        self.vae_adapter = VAEAdapter(self.vae)
        self.vae_optimizer = torch.optim.Adam(self.vae.parameters(), lr=0.001)

        # Initialize LSTM Agent (optional)
        self.use_lstm = config.get('lstm', {}).get('enabled', False)
        if self.use_lstm:
            self.lstm_agent = LSTMAgent(
                state_size=config['model']['state_size'],
                action_size=config['model']['action_size'],
                seq_len=config.get('lstm', {}).get('seq_len', 10)
            )
        else:
            self.lstm_agent = None

        # Initialize active learning
        self.active = ActiveLearningPipeline(
            uncert_thresh=config['active_learning']['uncertainty_threshold'],
            div_weight=config['active_learning']['diversity_weight'],
            budget=config['active_learning']['label_budget'],
            sim_thresh=config['active_learning']['similarity_threshold']
        )

        # Initialize LLM components
        self.llm_enabled = config.get('llm', {}).get('enabled', False)
        if self.llm_enabled:
            self.llm = LLMAdvisor(
                model=config['llm'].get('model', 'llama3.2:1b'),
                temperature=config['llm'].get('temperature', 0.3),
                timeout=60
            )
            self.shaper = RewardShaper(self.llm)
            self.al_advisor = ALAdvisor(self.llm)
        else:
            self.llm = None
            self.shaper = None
            self.al_advisor = None

        # Metrics
        self.step_count = 0
        self.rewards = []
        self.actions = []
        self.temperatures = []
        self.humidities = []
        self.statuses = []
        self.query_history = []
        self.llm_advice_count = 0
        self.vae_scores = []

        # For state tracking
        self.last_state = None
        self.last_action = None

    def step(self, data, idx=None):
        self.step_count += 1

        # Agent step
        result = self.agent.process(data)
        action = result['action']
        reward = result['reward']
        state = result['state']

        # VAE detection
        vae_result = self.vae_adapter.process(state)
        self.vae_scores.append(vae_result['vae_score'])
        result['vae'] = vae_result

        # Train VAE occasionally
        if self.step_count % 50 == 0:
            state_tensor = torch.FloatTensor(state).unsqueeze(0).to(next(self.vae.parameters()).device)
            vae_loss = self.vae.train_step(state_tensor, self.vae_optimizer)
            result['vae_loss'] = vae_loss

        # LSTM agent step if enabled
        if self.use_lstm and self.lstm_agent:
            lstm_action = self.lstm_agent.act(state, training=True)
            self.lstm_agent.update()
            result['lstm_action'] = lstm_action

        # Get LLM advice if enabled
        llm_result = None
        if self.llm_enabled and self.llm and self.step_count % 20 == 0:
            stats = self.get_stats()
            context = {
                'accuracy': stats.get('agent', {}).get('accuracy', 0),
                'precision': stats.get('agent', {}).get('precision', 0),
                'recall': stats.get('agent', {}).get('recall', 0),
                'f1_score': stats.get('agent', {}).get('f1_score', 0),
                'epsilon': stats.get('agent', {}).get('epsilon', 1.0),
                'queries_used': stats.get('active', {}).get('queries', 0),
                'budget': self.active.selector.budget,
                'pseudo_labels': stats.get('active', {}).get('pseudo_labels', 0),
                'alert_rate': stats.get('alert_rate', 0),
                'avg_reward': stats.get('avg_reward', 0),
                'steps': self.step_count,
                'vae_score': np.mean(self.vae_scores[-100:]) if self.vae_scores else 0
            }

            advice = self.llm.get_advice(context)
            if advice:
                self.llm_advice_count += 1
                self.agent.apply_llm_advice(advice)

                if self.al_advisor:
                    al_strategy, confidence = self.al_advisor.advise_strategy(stats.get('active', {}))
                    if al_strategy == 'more_queries':
                        self.active.selector.uncert_thresh = max(0.3, self.active.selector.uncert_thresh - 0.02)
                    elif al_strategy == 'more_propagation':
                        self.active.propagator.sim_thresh = min(0.95, self.active.propagator.sim_thresh + 0.01)

                llm_result = advice

        # Apply reward shaping if enabled
        if self.shaper and llm_result:
            stats = self.get_stats()
            context = {
                'accuracy': stats.get('agent', {}).get('accuracy', 0),
                'epsilon': self.agent.epsilon,
                'vae_score': vae_result['vae_score']
            }
            shaped_reward = self.shaper.shape(reward, state, action, context)
            result['final_reward'] = shaped_reward
            result['reward_shaped'] = True
        else:
            result['final_reward'] = reward
            result['reward_shaped'] = False

        # Store metrics
        self.rewards.append(result['final_reward'])
        self.actions.append(action)
        self.temperatures.append(data['temperature'])
        self.humidities.append(data['humidity'])
        self.statuses.append(data['status'])

        # Active Learning
        active_result = None
        if (self.step_count % 10 == 0 or idx is not None) and self.active.selector.queries < self.active.selector.budget:
            state_tensor = torch.FloatTensor(state).unsqueeze(0)

            with torch.no_grad():
                state_batch = torch.FloatTensor(state).unsqueeze(0).to(self.agent.policy_net.fc1.weight.device)
                qvals = self.agent.policy_net(state_batch)

            active_result = self.active.process(
                states=state_tensor,
                qvals=qvals.cpu(),
                actions=torch.tensor([action]),
                rewards=torch.tensor([result['final_reward']])
            )

            if active_result['propagated']:
                self.agent.apply_pseudo_labels(active_result['propagated'])

            if active_result['query_idx']:
                self.query_history.append({
                    'step': self.step_count,
                    'idx': idx if idx is not None else self.step_count,
                    'state': state.tolist(),
                    'action': action,
                    'temperature': data['temperature'],
                    'humidity': data['humidity'],
                    'status': data['status'],
                    'vae_score': vae_result['vae_score']
                })

        result['active'] = active_result
        result['llm'] = llm_result
        result['step'] = self.step_count

        return result

    def get_stats(self):
        if not self.rewards:
            return {}

        recent_window = min(100, len(self.rewards))
        recent_rewards = self.rewards[-recent_window:]
        recent_actions = self.actions[-recent_window:]

        alert_rate = sum(1 for a in recent_actions if a == 1) / len(recent_actions) if recent_actions else 0

        agent_stats = self.agent.get_stats()

        stats = {
            'steps': self.step_count,
            'avg_reward': np.mean(recent_rewards),
            'alert_rate': alert_rate,
            'total_alerts': sum(1 for a in self.actions if a == 1),
            'avg_temperature': np.mean(self.temperatures[-100:]) if self.temperatures else 0,
            'avg_humidity': np.mean(self.humidities[-100:]) if self.humidities else 0,
            'agent': agent_stats,
            'vae': {
                'avg_score': np.mean(self.vae_scores[-100:]) if self.vae_scores else 0,
                'latest_score': self.vae_scores[-1] if self.vae_scores else 0
            },
            'active': {
                'queries': self.active.selector.queries,
                'propagated': self.active.propagator.count,
                'pseudo_labels': len(self.active.pseudo_labels),
                'budget_remaining': self.active.selector.budget - self.active.selector.queries,
                'total_queries': len(self.query_history),
                'uncertainty_threshold': self.active.selector.uncert_thresh,
                'similarity_threshold': self.active.propagator.sim_thresh
            },
            'llm': {
                'enabled': self.llm_enabled,
                'advice_count': self.llm_advice_count
            }
        }

        if self.llm_enabled and self.llm:
            stats['llm'].update(self.llm.get_stats())

        return stats

    def save_models(self, prefix="models/"):
        os.makedirs(prefix, exist_ok=True)
        self.agent.save_model(f"{prefix}agent.pth")
        self.vae.save(f"{prefix}vae.pth")

        if self.use_lstm and self.lstm_agent:
            self.lstm_agent.save(f"{prefix}lstm_agent.pth")

        torch.save({
            'labeled': self.active.selector.labeled,
            'queries': self.active.selector.queries,
            'pseudo_labels': self.active.pseudo_labels,
            'propagator_count': self.active.propagator.count,
            'uncert_thresh': self.active.selector.uncert_thresh,
            'sim_thresh': self.active.propagator.sim_thresh
        }, f"{prefix}active_learning.pth")

        print(f"✅ All models saved to {prefix}")

    def load_models(self, prefix="models/"):
        self.agent.load_model(f"{prefix}agent.pth")
        self.vae.load(f"{prefix}vae.pth")

        if self.use_lstm and self.lstm_agent:
            self.lstm_agent.load(f"{prefix}lstm_agent.pth")

        al_path = f"{prefix}active_learning.pth"
        if os.path.exists(al_path):
            al_state = torch.load(al_path)
            self.active.selector.labeled = al_state.get('labeled', [])
            self.active.selector.queries = al_state.get('queries', 0)
            self.active.pseudo_labels = al_state.get('pseudo_labels', {})
            self.active.propagator.count = al_state.get('propagator_count', 0)
            self.active.selector.uncert_thresh = al_state.get('uncert_thresh', self.active.selector.uncert_thresh)
            self.active.propagator.sim_thresh = al_state.get('sim_thresh', self.active.propagator.sim_thresh)
            print("✅ Active learning state loaded")